# Phase 6 — Full-Scene Inference & Interactive Zoom Viewer

Pick any point on (or near) Lake Erie and a date, and see the trained model's real-time
land / no-bloom / bloom classification laid over the true-color Sentinel-2 image for that spot.

Unlike Stage 1's bulk pipeline (which exports large point samples to Drive and waits), this uses
`ee.Image.sampleRectangle()` to pull a small region's pixels **synchronously** -- no export task, no
waiting -- which is what makes a "click and see" interactive experience possible here. It only works
for small areas (a few km across); that's the right trade-off for an interactive viewer.

### What you need
- Your trained model checkpoint (`snn_bloom_classifier_tuned.pt`) with `selected_features`,
  `scaler_mean`/`scaler_scale`, and `config` saved inside it (same format Phase 4 produced).


## 1. Config

In [ ]:
GEE_PROJECT_ID = 'YOUR_GEE_PROJECT_ID'   # <-- your GEE project id
MODEL_PATH = '/content/snn_bloom_classifier_tuned.pt'

DEFAULT_RADIUS_KM = 1.2     # keep small -- sampleRectangle has a pixel-count ceiling
CLOUD_PCT_MAX = 40
COMPOSITE_WINDOW_DAYS = 5

# A few known-interesting starting points to try immediately
PRESETS = {
    'Western basin (recurring bloom hotspot)': (41.70, -83.25, '2024-09-03'),
    'Maumee Bay mouth': (41.70, -83.47, '2024-08-13'),
    'Central basin (typically calm)': (41.75, -81.50, '2024-06-20'),
    'Eastern basin (typically calm)': (42.30, -79.80, '2024-06-20'),
}

In [ ]:
# Install required packages
!pip install -q earthengine-api snntorch scikit-learn ipywidgets

In [ ]:
import ee
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from datetime import datetime, timedelta

import snntorch as snn
from snntorch import surrogate

import ipywidgets as widgets
from IPython.display import display, clear_output


# ============================================================
# PROJECT CONFIGURATION
# ============================================================

GEE_PROJECT_ID = 'YOUR_GEE_PROJECT_ID'

MODEL_PATH = '/content/snn_bloom_classifier_tuned.pt'

DEFAULT_RADIUS_KM = 1.2
CLOUD_PCT_MAX = 40
COMPOSITE_WINDOW_DAYS = 5


# ============================================================
# PRESET LOCATIONS
# ============================================================

PRESETS = {
    'Western basin (recurring bloom hotspot)':
        (41.70, -83.25, '2024-09-03'),

    'Maumee Bay mouth':
        (41.70, -83.47, '2024-08-13'),

    'Central basin (typically calm)':
        (41.75, -81.50, '2024-06-20'),

    'Eastern basin (typically calm)':
        (42.30, -79.80, '2024-06-20'),
}

print("Configuration loaded.")

In [ ]:
# ============================================================
# EARTH ENGINE AUTHENTICATION
# ============================================================

ee.Authenticate(auth_mode='notebook')

ee.Initialize(project=GEE_PROJECT_ID)

print("Earth Engine ready.")

## 2. Install, authenticate, imports

In [ ]:
!pip install -U earthengine-api snntorch scikit-learn ipywidgets -q

import ee
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from datetime import datetime, timedelta

import snntorch as snn
from snntorch import surrogate

import ipywidgets as widgets
from IPython.display import display, clear_output

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT_ID)
print('Earth Engine ready.')

## 3. Load the trained model

In [ ]:
class SNNClassifier(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_outputs, beta):
        super().__init__()
        spike_grad = surrogate.fast_sigmoid()
        self.fc1  = nn.Linear(num_inputs, num_hidden)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.fc2  = nn.Linear(num_hidden, num_outputs)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, x, num_steps):
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        spk2_rec = []
        for _ in range(num_steps):
            cur1 = self.fc1(x)
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            spk2_rec.append(spk2)
        return torch.stack(spk2_rec, dim=0)


checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
cfg = checkpoint['config']
selected_features = checkpoint['selected_features']
scaler_mean = np.array(checkpoint['scaler_mean'])
scaler_scale = np.array(checkpoint['scaler_scale'])

ALL_31_FEATURES = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12',
                    'NDWI','NDVI','MNDWI','GNDVI','NDCI','FAI','AFAI','SABI',
                    'CI_GREEN','CI_RED_EDGE','NDTI','NDMI','WATER_RATIO','TURBIDITY','FVI',
                    'NDCI_mean','NDCI_var','NDCI_std','NDCI_min','NDCI_max','NDCI_range']
assert len(ALL_31_FEATURES) == 31
selected_indices = [ALL_31_FEATURES.index(f) for f in selected_features]

model = SNNClassifier(cfg['num_inputs'], cfg['num_hidden'], cfg['num_outputs'], cfg['beta'])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print('Model loaded. Selected features:', selected_features)

## 4. Feature computation (identical formulas to Stage 1, applied to a whole image)

In [ ]:
EPS = 1e-6

def mask_s2_clouds(image):
    scl = image.select('SCL')
    clear = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9))
                        .And(scl.neq(10)).And(scl.neq(11)))
    return image.updateMask(clear)

def build_composite(lat, lon, radius_km, date_str, window_days=COMPOSITE_WINDOW_DAYS):
    center_date = datetime.strptime(date_str, '%Y-%m-%d')
    start = (center_date - timedelta(days=window_days)).strftime('%Y-%m-%d')
    end   = (center_date + timedelta(days=window_days + 1)).strftime('%Y-%m-%d')

    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(radius_km * 1000).bounds()

    coll = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(region)
            .filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_PCT_MAX))
            .map(mask_s2_clouds))
    n = coll.size().getInfo()
    if n == 0:
        return None, None, 0
    img = coll.select(['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']).median().divide(10000)
    return img.clip(region), region, n

def compute_feature_stack(img):
    B2, B3, B4  = img.select('B2'), img.select('B3'), img.select('B4')
    B5, B6, B7  = img.select('B5'), img.select('B6'), img.select('B7')
    B8, B8A     = img.select('B8'), img.select('B8A')
    B11, B12    = img.select('B11'), img.select('B12')

    NDWI  = img.normalizedDifference(['B3','B8']).rename('NDWI')
    NDVI  = img.normalizedDifference(['B8','B4']).rename('NDVI')
    MNDWI = img.normalizedDifference(['B3','B11']).rename('MNDWI')
    GNDVI = img.normalizedDifference(['B8','B3']).rename('GNDVI')
    NDCI  = img.normalizedDifference(['B5','B4']).rename('NDCI')

    FAI = img.expression(
        'B8 - (B4 + (B11 - B4) * ((842.0 - 665.0) / (1610.0 - 665.0)))',
        {'B8': B8, 'B4': B4, 'B11': B11}).rename('FAI')
    AFAI = img.expression(
        'B8A - (B4 + (B11 - B4) * ((865.0 - 665.0) / (1610.0 - 665.0)))',
        {'B8A': B8A, 'B4': B4, 'B11': B11}).rename('AFAI')
    SABI = B8.subtract(B4).divide(B2.add(B3).max(EPS)).rename('SABI')
    CI_GREEN = img.expression(
        'B3 - (B4 + (B5 - B4) * ((560.0 - 665.0) / (705.0 - 665.0)))',
        {'B3': B3, 'B4': B4, 'B5': B5}).rename('CI_GREEN')
    CI_RED_EDGE = img.expression(
        'B5 - (B4 + (B6 - B4) * ((705.0 - 665.0) / (740.0 - 665.0)))',
        {'B5': B5, 'B4': B4, 'B6': B6}).rename('CI_RED_EDGE')
    NDTI = img.normalizedDifference(['B4','B3']).rename('NDTI')
    NDMI = img.normalizedDifference(['B8','B11']).rename('NDMI')
    WATER_RATIO = B3.divide(B8.max(EPS)).rename('WATER_RATIO')
    TURBIDITY   = B4.divide(B3.max(EPS)).rename('TURBIDITY')
    FVI         = B8.divide(B4.max(EPS)).rename('FVI')

    index_stack = ee.Image.cat([
        NDWI, NDVI, MNDWI, GNDVI, NDCI, FAI, AFAI, SABI,
        CI_GREEN, CI_RED_EDGE, NDTI, NDMI, WATER_RATIO, TURBIDITY, FVI
    ])

    full_bands = img.addBands(index_stack)
    kernel = ee.Kernel.square(radius=1, units='pixels')
    ndci = index_stack.select('NDCI')
    NDCI_mean  = ndci.reduceNeighborhood(ee.Reducer.mean(), kernel).rename('NDCI_mean')
    NDCI_var   = ndci.reduceNeighborhood(ee.Reducer.variance(), kernel).rename('NDCI_var')
    NDCI_std   = ndci.reduceNeighborhood(ee.Reducer.stdDev(), kernel).rename('NDCI_std')
    NDCI_min   = ndci.reduceNeighborhood(ee.Reducer.min(), kernel).rename('NDCI_min')
    NDCI_max   = ndci.reduceNeighborhood(ee.Reducer.max(), kernel).rename('NDCI_max')
    NDCI_range = NDCI_max.subtract(NDCI_min).rename('NDCI_range')
    spatial_stack = ee.Image.cat([NDCI_mean, NDCI_var, NDCI_std, NDCI_min, NDCI_max, NDCI_range])

    feature_stack = full_bands.addBands(spatial_stack)
    assert len(feature_stack.bandNames().getInfo()) == 31
    return feature_stack, MNDWI

print('Feature functions ready.')

## 5. Core function — fetch, classify, and plot a small region synchronously
`sampleRectangle` pulls every pixel in the region directly into Python -- no Drive export, no
waiting. This is what makes an interactive "click a point, see the result" workflow possible; it
only scales to small areas (a few km across), which is exactly the right size for "zoom into one
spot."


In [ ]:
def classify_region(lat, lon, date_str, radius_km=DEFAULT_RADIUS_KM):
    print(f'Fetching Sentinel-2 around ({lat:.4f}, {lon:.4f}) for {date_str} (radius {radius_km} km)...')

    img, region, n_scenes = build_composite(lat, lon, radius_km, date_str)
    if img is None:
        print('No cloud-free Sentinel-2 scenes found in this window -- try a different date.')
        return None

    feature_stack, MNDWI = compute_feature_stack(img)
    true_color = img.select(['B4', 'B3', 'B2'])  # RGB
    combined = feature_stack.addBands(true_color.rename(['RGB_R', 'RGB_G', 'RGB_B']))

    sample = combined.sampleRectangle(region=region, defaultValue=-9999)
    band_names = combined.bandNames().getInfo()
    band_arrays = {}
    for b in band_names:
        arr = np.array(sample.get(b).getInfo(), dtype=np.float32)
        band_arrays[b] = arr

    shape = band_arrays[band_names[0]].shape
    print(f'Fetched a {shape[0]} x {shape[1]} pixel grid ({n_scenes} scene(s) composited).')

    feature_grid = np.stack([band_arrays[f] for f in ALL_31_FEATURES], axis=-1)   # (H, W, 31)
    water_mask = band_arrays['MNDWI'] > 0
    rgb = np.stack([band_arrays['RGB_R'], band_arrays['RGB_G'], band_arrays['RGB_B']], axis=-1)
    rgb = np.clip(rgb * 3.5, 0, 1)   # simple brightness stretch for viewing

    H, W, _ = feature_grid.shape
    flat = feature_grid.reshape(-1, 31)
    valid = ~np.any(flat <= -9990, axis=1)   # drop no-data pixels (outside composite coverage)

    scaled = np.zeros_like(flat)
    scaled[valid] = (flat[valid] - scaler_mean) / scaler_scale
    scaled_sel = scaled[:, selected_indices]

    probs_flat = np.zeros(flat.shape[0], dtype=np.float32)
    with torch.no_grad():
        Xt = torch.tensor(scaled_sel[valid], dtype=torch.float32)
        spk_rec = model(Xt, cfg['num_steps'])
        probs = torch.softmax(spk_rec.sum(dim=0), dim=1)[:, 1].numpy()
        probs_flat[valid] = probs

    prob_grid = probs_flat.reshape(H, W)
    water_flat = water_mask.reshape(-1)

    return {
        'rgb': rgb, 'prob_grid': prob_grid, 'water_mask': water_mask,
        'valid': valid.reshape(H, W), 'lat': lat, 'lon': lon, 'date': date_str,
    }

## 6. Plot: true-color image next to the classified overlay

In [ ]:
def plot_result(result, threshold=0.5):
    if result is None:
        return
    rgb, prob_grid, water_mask, valid = result['rgb'], result['prob_grid'], result['water_mask'], result['valid']

    # Build a 3-class display: land (gray), no-bloom water (green), bloom water (red)
    display_map = np.zeros((*prob_grid.shape, 3))
    land = ~water_mask
    no_bloom = water_mask & (prob_grid < threshold)
    bloom = water_mask & (prob_grid >= threshold)

    display_map[land] = [0.75, 0.75, 0.75]
    display_map[no_bloom] = [0.20, 0.55, 0.30]
    display_map[bloom] = [0.80, 0.10, 0.10]
    display_map[~valid] = [1, 1, 1]

    fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
    axes[0].imshow(rgb)
    axes[0].set_title(f"True Color -- {result['date']}")
    axes[0].axis('off')

    axes[1].imshow(display_map)
    axes[1].set_title('Classification: gray=land, green=no-bloom, red=bloom')
    axes[1].axis('off')

    fig.suptitle(f"Lake Erie zoom: ({result['lat']:.4f}, {result['lon']:.4f})", fontsize=12)
    plt.tight_layout()
    plt.show()

    n_water = water_mask.sum()
    n_bloom = bloom.sum()
    if n_water > 0:
        print(f'Water pixels in view: {n_water}  |  Classified as bloom: {n_bloom} '
              f'({100*n_bloom/n_water:.1f}% of water area)')
    else:
        print('No water detected in this view -- try a different location.')

## 7. Try the presets directly

In [ ]:
for name, (lat, lon, date_str) in PRESETS.items():
    print(f'=== {name} ===')
    result = classify_region(lat, lon, date_str)
    plot_result(result)
    print()

## 8. Interactive widget — pick any point yourself
Enter coordinates and a date, or choose a preset, then click **Classify**.

In [ ]:
lat_box = widgets.FloatText(value=41.70, description='Latitude:', step=0.01)
lon_box = widgets.FloatText(value=-83.25, description='Longitude:', step=0.01)
date_box = widgets.Text(value='2024-09-03', description='Date:')
radius_box = widgets.FloatSlider(value=DEFAULT_RADIUS_KM, min=0.5, max=3.0, step=0.1, description='Radius (km):')
preset_dropdown = widgets.Dropdown(options=list(PRESETS.keys()), description='Preset:')
run_button = widgets.Button(description='Classify', button_style='success')
output = widgets.Output()

def on_preset_change(change):
    lat, lon, date_str = PRESETS[change['new']]
    lat_box.value, lon_box.value, date_box.value = lat, lon, date_str

preset_dropdown.observe(on_preset_change, names='value')

def on_run_clicked(b):
    with output:
        clear_output(wait=True)
        result = classify_region(lat_box.value, lon_box.value, date_box.value, radius_box.value)
        plot_result(result)

run_button.on_click(on_run_clicked)

display(widgets.VBox([
    preset_dropdown,
    widgets.HBox([lat_box, lon_box]),
    widgets.HBox([date_box, radius_box]),
    run_button,
    output,
]))

## Notes and honest limitations
- **Coverage**: this only works where Sentinel-2 + a cloud-free composite is available for the
  date/window you pick -- cloudy dates will return "no scenes found."
- **Radius ceiling**: `sampleRectangle` pulls the whole region into memory at once; keep the radius
  small (under ~3km) or it will be slow or hit Earth Engine's pixel-count limit.
- **Geographic validity**: this model is tuned to Lake Erie's water chemistry (see the
  Limitations slide) -- pointing it at a completely different lake will run without error but the
  classification should not be trusted without local validation, exactly as discussed for the
  Indian-lakes question earlier.
